# LightFM：算法、评估与推荐

先运行 `python train.py --model lightfm --loss logistic`。本 Notebook 展示选定模型，并与 ItemCF 使用相同用户和候选规则比较。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
                    if (path / 'src' / 'models' / 'lightfm.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.recommend import file_sha256, recommend_for_user

TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
run_path = TABLE_DIR / 'lightfm_run.json'
if not run_path.is_file():
    raise FileNotFoundError('请先运行 python train.py --model lightfm。')
run = json.loads(run_path.read_text(encoding='utf-8'))
for name, checksum in run['input_sha256'].items():
    if file_sha256(PROJECT_ROOT / 'data' / 'processed' / name) != checksum:
        raise ValueError('预处理输入已变化，请重新训练 LightFM。')

## 1. 算法与当前参数

当前 LightFM 只使用用户和电影 ID 特征，为每个 ID 学习独立的 32 维向量和偏置，通过内积与偏置评分。不输入人口属性、电影属性或历史统计特征。

采用 logistic：评分 ≥4 为 +1，1–2 分为 -1，3 分和未评分项不参与拟合。为保持固定拟合样本口径，参数学习仍使用训练内部时间划分的后半段标签；仅由训练数据更新参数。

正则化 1e-6、学习率 0.05、种子 42；比较第 5、10 轮，按验证 NDCG@10 选择第 10 轮。测试期使用固定用户向量，验证历史用于过滤已评分电影。


In [2]:
display(pd.DataFrame([run['selected_parameters']]))
search = pd.read_csv(TABLE_DIR / 'lightfm_validation_search.csv')
display(search[['epochs', 'Precision@10', 'Recall@10', 'NDCG@10', 'selected']]
        .style.format({'Precision@10': '{:.2%}', 'Recall@10': '{:.2%}', 'NDCG@10': '{:.4f}'}))


,loss,user_features,item_features,no_components,alpha,epochs
0,logistic,id_only,id_only,32,0.000001,10


,epochs,Precision@10,Recall@10,NDCG@10,selected
0,5.000000,3.85%,4.20%,0.0488,False
1,10.000000,3.99%,4.04%,0.0497,True


## 2. 与 ItemCF 比较

两种算法使用相同数据划分、评估用户和完整未评分候选目录。各指标先按用户计算再平均：
Precision@10＝命中数 /10，Recall@10＝命中数 /高分目标数，NDCG@10 同时考虑命中位置。
未评分电影不计为已知命中，但其偏好未知。LightFM 的用户向量仅由训练集学习，验证历史在测试时用于过滤已评分电影。

In [3]:
comparison = pd.read_csv(TABLE_DIR / 'comparison_metrics.csv')
comparison = comparison[comparison.algorithm.isin(['ItemCF', 'LightFM'])]
display(comparison.fillna({'loss': '-'})[['algorithm', 'loss', 'split', 'users', 'Precision@10', 'Recall@10', 'NDCG@10']]
        .style.format({'Precision@10': '{:.2%}', 'Recall@10': '{:.2%}',
                       'NDCG@10': '{:.4f}'}))

,algorithm,loss,split,users,Precision@10,Recall@10,NDCG@10
0,ItemCF,-,validation,5776,6.00%,7.91%,0.0847
1,ItemCF,-,test,5765,5.68%,7.18%,0.0779
2,LightFM,logistic,validation,5776,3.99%,4.04%,0.0497
3,LightFM,logistic,test,5765,3.92%,4.01%,0.0486


## 3. 按用户比较推荐电影

修改 `USER_ID` 后运行下面的单元格。每种算法推荐 5 部历史未评分电影，展示标题与类型。

In [4]:
USER_ID = 1
TOP_K = 5

recommendations = []
for algorithm in ['itemcf', 'lightfm']:
    result = recommend_for_user(USER_ID, stage='test', k=TOP_K, algorithm=algorithm)
    result.insert(0, 'algorithm', algorithm)
    recommendations.append(result[['algorithm', 'rank', 'title', 'genres']])
display(pd.concat(recommendations, ignore_index=True))

,algorithm,rank,title,genres
0,itemcf,1,Raiders of the Lost Ark (1981),Action|Adventure
1,itemcf,2,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
2,itemcf,3,"Shawshank Redemption, The (1994)",Drama
3,itemcf,4,"Silence of the Lambs, The (1991)",Drama|Thriller
4,itemcf,5,Pulp Fiction (1994),Crime|Drama
5,lightfm,1,Raiders of the Lost Ark (1981),Action|Adventure
6,lightfm,2,"Shawshank Redemption, The (1994)",Drama
7,lightfm,3,"Silence of the Lambs, The (1991)",Drama|Thriller
8,lightfm,4,"Godfather, The (1972)",Action|Crime|Drama
9,lightfm,5,"Fugitive, The (1993)",Action|Thriller


算法参考：Kula (2015), [Metadata Embeddings for User and Item Cold-start Recommendations](https://arxiv.org/abs/1507.08439)。训练使用[官方 LightFM 实现](https://making.lyst.com/lightfm/docs/lightfm.html)。